In [3]:
#!/usr/bin/env python3
import os
import json
import shutil
from pathlib import Path
from tempfile import TemporaryDirectory

import torch
from huggingface_hub import HfApi, upload_folder
from transformers import AutoConfig, AutoModelForCausalLM

# 1) Create a tiny GPT-NeoX backbone config
base_cfg = AutoConfig.for_model(
    "gpt_neox",
    hidden_size=128,
    num_hidden_layers=4,
    num_attention_heads=4,
    intermediate_size=512,
    max_position_embeddings=512,
)

# 2) Build ARMT model from config (uses local project code)
from modeling_amt.model import ARMTForCausalLM, ARMTConfig

armt_cfg = ARMTConfig(
    base_model_config=base_cfg,
    num_mem_tokens=16,
    d_mem=32,
    segment_size=128,              # arbitrary small segment size
    segment_alignment="left",
    sliding_window=False,
    layers_attr="gpt_neox.layers", # GPT-NeoX layers path
    wrap_pos=False,
    correction=True,
    n_heads=1,
    use_denom=True,
    gating=False,
    freeze_mem=False,
    act_on=True,
    max_hop=4,
    act_type="layer",              # same as in CA scripts
    act_format="linear",
    noisy_halting=False,
    constant_depth=False,
    time_penalty=0.0,
)
model = ARMTForCausalLM(armt_cfg)
model.eval()

# 3) Prepare a local repo folder with weights, config, and custom code
repo_id = "irodkin/armt-act-layer-neox-tiny"

local_modeling_dir = Path("./modeling_amt")

with TemporaryDirectory() as tmpdir:
    repo_dir = Path(tmpdir) / "repo"
    repo_dir.mkdir(parents=True, exist_ok=True)

    # Save model weights and config
    model.save_pretrained(repo_dir)

    # Ensure auto_map for remote-code loading
    cfg_path = repo_dir / "config.json"
    cfg = json.loads(cfg_path.read_text())
    cfg["architectures"] = ["ARMTForCausalLM"]
    cfg["auto_map"] = {
        "AutoConfig": "armt_entry.ARMTConfig",
        "AutoModelForCausalLM": "armt_entry.ARMTForCausalLM",
    }
    cfg_path.write_text(json.dumps(cfg, indent=2))

    # Include minimal custom code package (modeling_amt/)
    remote_pkg_dir = repo_dir / "modeling_amt"
    remote_pkg_dir.mkdir(parents=True, exist_ok=True)
    (remote_pkg_dir / "__init__.py").write_text("")  # make it a package

    # Copy the two modules used by ARMTForCausalLM
    shutil.copy2(local_modeling_dir / "model.py", remote_pkg_dir / "model.py")
    shutil.copy2(local_modeling_dir / "language_modeling.py", remote_pkg_dir / "language_modeling.py")

    (repo_dir / "armt_entry.py").write_text(
        "from modeling_amt.model import ARMTForCausalLM, ARMTConfig\n"
    )

    # Optional: add a basic README
    (repo_dir / "README.md").write_text(
        f"# {repo_id}\n\nTiny GPT-NeoX + ARMT model for testing. Load with trust_remote_code=True."
    )

    # 4) Create/overwrite repo and upload everything
    api = HfApi()
    api.create_repo(repo_id, private=True, exist_ok=True)
    upload_folder(
        folder_path=str(repo_dir),
        repo_id=repo_id,
        repo_type="model",
        commit_message="Initial tiny ARMT (GPT-NeoX) upload",
    )


/home/ivan.rodkin/miniconda3/envs/env/lib/python3.9/site-packages/huggingface_hub/hf_api.py:9696: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")


model.safetensors:   0%|          | 0.00/55.2M [00:00<?, ?B/s]

In [4]:

# 5) Load back from the hub (uses remote code)
loaded = AutoModelForCausalLM.from_pretrained(repo_id, trust_remote_code=True)
print(type(loaded))
print("Loaded OK from Hub")

config.json:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

armt_entry.py:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/irodkin/armt-act-layer-neox-tiny:
- armt_entry.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/55.2M [00:00<?, ?B/s]

<class 'modeling_amt.model.ARMTForCausalLM'>
Loaded OK from Hub


In [5]:
loaded

ARMTForCausalLM(
  (armt): AssociativeRecurrentWrapper(
    (memory_cell): AssociativeMemoryCell(
      (model): GPTNeoXForCausalLM(
        (gpt_neox): GPTNeoXModel(
          (embed_in): Embedding(50432, 128)
          (emb_dropout): Dropout(p=0.0, inplace=False)
          (layers): ModuleList(
            (0-3): 4 x AdaptiveAssociativeLayerWrapper2(
              (W_mq): Linear(in_features=128, out_features=32, bias=False)
              (W_mk): Linear(in_features=128, out_features=32, bias=False)
              (W_mv): Linear(in_features=128, out_features=128, bias=False)
              (W_mb): Linear(in_features=128, out_features=1, bias=True)
              (layer): GPTNeoXLayer(
                (input_layernorm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
                (post_attention_layernorm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
                (post_attention_dropout): Dropout(p=0.0, inplace=False)
                (post_mlp_dropout): Dropout(p=0.0,